In [ ]:
import os
import json
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import LinearSegmentedColormap
import warnings

warnings.filterwarnings('ignore')

# Configure matplotlib
plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 100,
})

# Color schemes
HEATMAP_COLORS = ['#457B9D', '#A8DADC', '#F1FAEE', '#F8D7DA', '#E63946']
HEATMAP_CMAP = LinearSegmentedColormap.from_list('pastel_heatmap', HEATMAP_COLORS)
BAR_COLORS = ['#8FBC8F', '#F0B7A4', '#B8A9C9', '#A8E6CF']

# Model definitions
SOURCE_MODELS = [
    'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
    '_medcalc-deepseek-r1-distill-qwen-1.5b-inst-grpo'
]

TARGET_MODELS = [
    'openai_gpt-oss-20b',
    'BytedTsinghua-SIA_DAPO-Qwen-32B',
    'Qwen_QwQ-32B',
    'open-thoughts_OpenThinker-7B',
    'nvidia_Nemotron-Research-Reasoning-Qwen-1.5B'
]

MODEL_NAMES = SOURCE_MODELS + TARGET_MODELS


def create_short_model_names():
    """Create mapping for short model names."""
    return {
        'openai_gpt-oss-20b': 'OSS',
        'BytedTsinghua-SIA_DAPO-Qwen-32B': 'DAPO',
        'Qwen_QwQ-32B': 'QwQ',
        'open-thoughts_OpenThinker-7B': 'OpenT',
        'nvidia_Nemotron-Research-Reasoning-Qwen-1.5B': 'NRR',
        'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B': 'DeepSeek',
        '_medcalc-deepseek-r1-distill-qwen-1.5b-inst-grpo': 'DeepSeek-GRPO',
    }


def identify_model_types(sources):
    """Identify GRPO and original models."""
    grpo_model = next((s for s in sources if 'grpo' in s.lower()), None)
    orig_model = next((s for s in sources if 'grpo' not in s.lower()), None)
    return grpo_model, orig_model


def parse_jsonl_file(filepath):
    """Parse JSONL file and return list of data points."""
    data_points = []
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    data_points.append(json.loads(line))
                except json.JSONDecodeError as e:
                    print(f"Error parsing line {line_num} in {filepath}: {e}")
    except FileNotFoundError:
        print(f"File not found: {filepath}")
    except Exception as e:
        print(f"Error reading file {filepath}: {e}")
    return data_points


def extract_model_names_and_condition(filename):
    """Extract source, target model names and answer condition from filename."""
    pattern = r'(.+)_thoughts_to_(.+)_zero_shot'
    match = re.match(pattern, filename)
    if match:
        condition = "without" if "without_answer" in filename else "with"
        return match.group(1), match.group(2), condition
    return None, None, None


def check_consistency(dp1, dp2):
    """Check consistency between two data points."""
    ans1 = dp1.get("Target Answer", "")
    res1 = dp1.get("Target Result", "")
    ans2 = dp2.get("Target Answer", "")
    res2 = dp2.get("Target Result", "")
    
    invalid = ["not defined", "N/A", "does not match", "are not permitted"]
    if any(p in ans1 or p in ans2 for p in invalid):
        return None
    
    return 1 if ans1 == ans2 or (res1 == "Correct" and res2 == "Correct") else 0


def load_data(folder_path, filter_source_models=None, filter_target_models=None):
    """Load and organize data from JSONL files."""
    if not os.path.exists(folder_path):
        print(f"Folder path does not exist: {folder_path}")
        return None
    
    data_by_source = {'with': {}, 'without': {}}
    model_names = set()
    
    jsonl_files = [f for f in os.listdir(folder_path)
                   if f.endswith('.jsonl') and '_thoughts_to_' in f]
    
    if not jsonl_files:
        print("No JSONL files found matching the expected pattern.")
        return None
    
    name_mapping = create_short_model_names()
    print(f"\n**Filtering for:**")
    print(f"  Source models: {[name_mapping.get(m, m) for m in filter_source_models]}")
    print(f"  Target models: {[name_mapping.get(m, m) for m in filter_target_models]}")
    print(f"  Conditions: with_answer, without_answer\n")
    
    for filename in jsonl_files:
        source_model, target_model, condition = extract_model_names_and_condition(filename)
        
        if not (source_model and target_model and condition):
            continue
        
        if filter_source_models and source_model not in filter_source_models:
            continue
        if filter_target_models and target_model not in filter_target_models:
            continue
        
        model_names.add(source_model)
        model_names.add(target_model)
        
        filepath = os.path.join(folder_path, filename)
        data_points = parse_jsonl_file(filepath)
        
        if data_points:
            data_by_source[condition][(source_model, target_model)] = data_points
            short_source = name_mapping.get(source_model, source_model[:10])
            short_target = name_mapping.get(target_model, target_model[:10])
            print(f"{short_source} -> {short_target} [{condition}_answer]: "
                  f"Loaded {len(data_points)} data points")
    
    return data_by_source, sorted(list(model_names))


def calculate_cross_source_consistency(sources, data_by_source, name_mapping):
    """Calculate consistency between the two source models directly."""
    pairwise_consistency = {}
    
    if len(sources) != 2:
        return pairwise_consistency
    
    source_a, source_b = sources[0], sources[1]
    
    # Find common targets
    targets_a = [target for (src, target) in data_by_source.keys() if src == source_a]
    targets_b = [target for (src, target) in data_by_source.keys() if src == source_b]
    common_targets = set(targets_a) & set(targets_b)
    
    if not common_targets:
        return pairwise_consistency
    
    reference_target = sorted(list(common_targets))[0]
    reference_data_a = data_by_source[(source_a, reference_target)]
    reference_data_b = data_by_source[(source_b, reference_target)]
    
    consistent_pairs = 0
    total_pairs = 0
    
    for idx in range(min(len(reference_data_a), len(reference_data_b))):
        dp_a = {
            "Target Answer": reference_data_a[idx].get("LLM Answer", ""),
            "Target Result": reference_data_a[idx].get("Result", "")
        }
        dp_b = {
            "Target Answer": reference_data_b[idx].get("LLM Answer", ""),
            "Target Result": reference_data_b[idx].get("Result", "")
        }
        
        result = check_consistency(dp_a, dp_b)
        if result is not None:
            total_pairs += 1
            consistent_pairs += result
    
    if total_pairs > 0:
        pairwise_rate = consistent_pairs / total_pairs
        # Add to both sources for symmetry
        for source in [source_a, source_b]:
            pairwise_consistency[(source, source_a, source_b)] = {
                'consistency_rate': pairwise_rate,
                'consistent_pairs': consistent_pairs,
                'total_pairs': total_pairs
            }
        
        short_a = name_mapping.get(source_a, source_a[:10])
        short_b = name_mapping.get(source_b, source_b[:10])
        print(f"  Cross-source: {short_a} vs {short_b} -> "
              f"{pairwise_rate:.3f} ({consistent_pairs}/{total_pairs})")
    
    return pairwise_consistency


def calculate_pairwise_consistency(data_by_source):
    """Calculate pairwise consistency between all models for each source model."""
    pairwise_consistency = {}
    name_mapping = create_short_model_names()
    
    sources = sorted(set(src for (src, _) in data_by_source.keys()))
    
    # Add cross-source comparison
    cross_source = calculate_cross_source_consistency(sources, data_by_source, name_mapping)
    pairwise_consistency.update(cross_source)
    
    # Calculate pairwise consistency for each source model
    for source_model in sources:
        targets = [target for (src, target) in data_by_source.keys() if src == source_model]
        all_models = [source_model] + targets
        
        if len(all_models) < 2:
            continue
        
        reference_key = (source_model, targets[0])
        reference_data = data_by_source[reference_key]
        
        # Compare all pairs of models
        for i in range(len(all_models)):
            for j in range(i + 1, len(all_models)):
                model_a = all_models[i]
                model_b = all_models[j]
                
                # Skip if this is a cross-source pair (already handled)
                if model_a in sources and model_b in sources:
                    continue
                
                consistent_pairs = 0
                total_pairs = 0
                
                for idx in range(len(reference_data)):
                    dp_a = {}
                    dp_b = {}
                    
                    # Get data for model_a
                    if model_a == source_model:
                        dp_a["Target Answer"] = reference_data[idx].get("LLM Answer", "")
                        dp_a["Target Result"] = reference_data[idx].get("Result", "")
                    else:
                        key_a = (source_model, model_a)
                        if key_a in data_by_source and idx < len(data_by_source[key_a]):
                            dp_a["Target Answer"] = data_by_source[key_a][idx].get("Target Answer", "")
                            dp_a["Target Result"] = data_by_source[key_a][idx].get("Target Result", "")
                        else:
                            continue
                    
                    # Get data for model_b
                    if model_b == source_model:
                        dp_b["Target Answer"] = reference_data[idx].get("LLM Answer", "")
                        dp_b["Target Result"] = reference_data[idx].get("Result", "")
                    else:
                        key_b = (source_model, model_b)
                        if key_b in data_by_source and idx < len(data_by_source[key_b]):
                            dp_b["Target Answer"] = data_by_source[key_b][idx].get("Target Answer", "")
                            dp_b["Target Result"] = data_by_source[key_b][idx].get("Target Result", "")
                        else:
                            continue
                    
                    result = check_consistency(dp_a, dp_b)
                    if result is not None:
                        total_pairs += 1
                        consistent_pairs += result
                
                if total_pairs > 0:
                    pairwise_rate = consistent_pairs / total_pairs
                    pairwise_consistency[(source_model, model_a, model_b)] = {
                        'consistency_rate': pairwise_rate,
                        'consistent_pairs': consistent_pairs,
                        'total_pairs': total_pairs
                    }
                    
                    short_source = name_mapping.get(source_model, source_model[:10])
                    short_a = name_mapping.get(model_a, model_a[:10])
                    short_b = name_mapping.get(model_b, model_b[:10])
                    print(f"  {short_source}: {short_a} vs {short_b} -> "
                          f"{pairwise_rate:.3f} ({consistent_pairs}/{total_pairs})")
    
    return pairwise_consistency


def organize_pairwise_data(pairwise_data):
    """Organize pairwise data by source model with standardized pair names."""
    name_mapping = create_short_model_names()
    data_by_source = defaultdict(list)
    
    for (source, target_a, target_b), data in pairwise_data.items():
        sorted_targets = sorted([target_a, target_b])
        short_a = name_mapping.get(sorted_targets[0], sorted_targets[0][:10])
        short_b = name_mapping.get(sorted_targets[1], sorted_targets[1][:10])
        pair_name = f"{short_a}-{short_b}"
        
        data_by_source[source].append({
            'pair': pair_name,
            'consistency': data['consistency_rate'],
            'count': data['consistent_pairs'],
            'total': data['total_pairs']
        })
    
    return data_by_source


def plot_comprehensive_comparison(pairwise_with, pairwise_without, save_plots=False, pdf_pages=None):
    """Create comprehensive comparison across all dimensions."""
    name_mapping = create_short_model_names()
    
    data_with = organize_pairwise_data(pairwise_with)
    data_without = organize_pairwise_data(pairwise_without)
    
    # Get all unique pairs and sources
    all_pairs = set()
    for source_data in list(data_with.values()) + list(data_without.values()):
        for item in source_data:
            all_pairs.add(item['pair'])
    all_pairs = sorted(list(all_pairs))
    
    sources = sorted(set(list(data_with.keys()) + list(data_without.keys())))
    grpo_model, orig_model = identify_model_types(sources)
    
    # Create figure
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
    
    # Plot 1: Full comparison
    ax1 = fig.add_subplot(gs[0, :])
    x = np.arange(len(all_pairs))
    width = 0.2
    
    for src_idx, source in enumerate(sources):
        short_source = name_mapping.get(source, source[:10])
        
        with_dict = {item['pair']: item['consistency'] for item in data_with.get(source, [])}
        without_dict = {item['pair']: item['consistency'] for item in data_without.get(source, [])}
        
        with_values = [with_dict.get(pair, 0) for pair in all_pairs]
        without_values = [without_dict.get(pair, 0) for pair in all_pairs]
        
        offset_with = (src_idx * 2 - 1.5) * width
        offset_without = (src_idx * 2 - 0.5) * width
        
        ax1.bar(x + offset_with, with_values, width,
                label=f'{short_source} (with answer)',
                color=BAR_COLORS[src_idx], alpha=0.9,
                edgecolor='white', linewidth=1)
        ax1.bar(x + offset_without, without_values, width,
                label=f'{short_source} (without answer)',
                color=BAR_COLORS[src_idx], alpha=0.5,
                edgecolor='white', linewidth=1, hatch='//')
    
    ax1.set_xlabel('Model Pairs', fontweight='bold', fontsize=11)
    ax1.set_ylabel('Pairwise Consistency Rate', fontweight='bold', fontsize=11)
    ax1.set_title('Pairwise Consistency: Source Model × Answer Condition',
                  fontweight='bold', pad=15, fontsize=13)
    ax1.set_xticks(x)
    ax1.set_xticklabels(all_pairs, rotation=45, ha='right')
    ax1.legend(frameon=True, fancybox=True, shadow=False, facecolor='white', ncol=2)
    ax1.grid(axis='y', alpha=0.3, linestyle='--', color='gray')
    ax1.set_ylim(0, 1.0)
    
    # Plot 2: Effect of answer inclusion
    ax2 = fig.add_subplot(gs[1, 0])
    
    for src_idx, source in enumerate(sources):
        short_source = name_mapping.get(source, source[:10])
        
        with_dict = {item['pair']: item['consistency'] for item in data_with.get(source, [])}
        without_dict = {item['pair']: item['consistency'] for item in data_without.get(source, [])}
        
        differences = []
        valid_pairs = []
        
        for pair in all_pairs:
            w = with_dict.get(pair)
            wo = without_dict.get(pair)
            if w and wo:
                differences.append(w - wo)
                valid_pairs.append(pair)
        
        if differences:
            y_pos = np.arange(len(valid_pairs)) + src_idx * 0.4
            bar_colors = ['#2D5016' if d > 0 else '#E63946' if d < 0 else '#F1FAEE'
                         for d in differences]
            
            ax2.barh(y_pos, differences, height=0.35, label=short_source,
                    color=bar_colors, alpha=0.8, edgecolor='white', linewidth=1)
    
    ax2.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax2.set_ylabel('Model Pairs', fontweight='bold')
    ax2.set_xlabel('Difference (with - without answer)', fontweight='bold')
    ax2.set_title('Effect of Including Answer in Thoughts', fontweight='bold', pad=15)
    ax2.set_yticks(np.arange(len(all_pairs)) + 0.2)
    ax2.set_yticklabels(all_pairs)
    ax2.grid(axis='x', alpha=0.3, linestyle='--', color='gray')
    
    # Plot 3: Effect of GRPO
    ax3 = fig.add_subplot(gs[1, 1])
    
    if grpo_model and orig_model:
        for cond_idx, (condition, data_dict) in enumerate([('with', data_with), ('without', data_without)]):
            dict_grpo = {item['pair']: item['consistency']
                        for item in data_dict.get(grpo_model, [])}
            dict_orig = {item['pair']: item['consistency']
                        for item in data_dict.get(orig_model, [])}
            
            differences = []
            valid_pairs = []
            
            for pair in all_pairs:
                val_grpo = dict_grpo.get(pair)
                val_orig = dict_orig.get(pair)
                if val_grpo and val_orig:
                    differences.append(val_grpo - val_orig)
                    valid_pairs.append(pair)
            
            if differences:
                y_pos = np.arange(len(valid_pairs)) + cond_idx * 0.4
                bar_colors = ['#2D5016' if d > 0 else '#E63946' if d < 0 else '#F1FAEE'
                             for d in differences]
                
                alpha = 0.9 if condition == 'with' else 0.5
                hatch = None if condition == 'with' else '//'
                
                ax3.barh(y_pos, differences, height=0.35, label=f'{condition} answer',
                        color=bar_colors, alpha=alpha, edgecolor='white',
                        linewidth=1, hatch=hatch)
    
    ax3.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax3.set_ylabel('Model Pairs', fontweight='bold')
    ax3.set_xlabel('Difference (GRPO - Original)', fontweight='bold')
    ax3.set_title('Effect of GRPO Training', fontweight='bold', pad=15)
    ax3.set_yticks(np.arange(len(all_pairs)) + 0.2)
    ax3.set_yticklabels(all_pairs)
    ax3.legend(frameon=True, fancybox=True, shadow=False, facecolor='white')
    ax3.grid(axis='x', alpha=0.3, linestyle='--', color='gray')
    
    plt.tight_layout()
    
    if save_plots:
        plt.savefig('llama_comprehensive_comparison.png', dpi=300,
                   bbox_inches='tight', facecolor='white')
        plt.savefig('llama_comprehensive_comparison.pdf',
                   bbox_inches='tight', facecolor='white')
    if pdf_pages:
        pdf_pages.savefig(fig, bbox_inches='tight', facecolor='white')
    plt.show()


def print_detailed_comparison(pairwise_with, pairwise_without):
    """Print detailed comparison across all dimensions."""
    name_mapping = create_short_model_names()
    
    print("\n" + "="*100)
    print("COMPREHENSIVE PAIRWISE CONSISTENCY COMPARISON")
    print("="*100)
    
    def organize_for_printing(pairwise_data):
        data_by_source = defaultdict(dict)
        for (source, target_a, target_b), data in pairwise_data.items():
            short_a = name_mapping.get(target_a, target_a[:10])
            short_b = name_mapping.get(target_b, target_b[:10])
            if short_a > short_b:
                short_a, short_b = short_b, short_a
            pair_name = f"{short_a} vs {short_b}"
            data_by_source[source][pair_name] = data
        return data_by_source
    
    data_with = organize_for_printing(pairwise_with)
    data_without = organize_for_printing(pairwise_without)
    
    all_pairs = set()
    for pairs_dict in list(data_with.values()) + list(data_without.values()):
        all_pairs.update(pairs_dict.keys())
    all_pairs = sorted(list(all_pairs))
    
    sources = sorted(set(list(data_with.keys()) + list(data_without.keys())))
    grpo_model, orig_model = identify_model_types(sources)
    
    # Print tables
    for condition_name, data_dict in [('WITH ANSWER', data_with), ('WITHOUT ANSWER', data_without)]:
        print(f"\n{condition_name}")
        print("-" * 100)
        print(f"{'Model Pair':<30}", end='')
        
        for source in sources:
            short_source = name_mapping.get(source, source[:20])
            print(f"{short_source:>20}", end='')
        
        if grpo_model and orig_model:
            print(f"{'Diff (GRPO-Orig)':>20}", end='')
        print()
        print("-" * 100)
        
        for pair in all_pairs:
            print(f"{pair:<30}", end='')
            values = {}
            
            pair_models = pair.replace(" vs ", "|").split("|")
            pair_includes_source = any(
                name_mapping.get(source, source[:10]) in pair_models
                for source in sources
            )
            
            for source in sources:
                short_source = name_mapping.get(source, source[:10])
                should_show = (not pair_includes_source or short_source in pair_models)
                
                if should_show and pair in data_dict[source]:
                    consistency = data_dict[source][pair]['consistency_rate']
                    values[source] = consistency
                    print(f"{consistency:>20.3f}", end='')
                else:
                    values[source] = None
                    print(f"{'N/A':>20}", end='')
            
            if (grpo_model and orig_model and
                grpo_model in values and orig_model in values and
                values[grpo_model] is not None and values[orig_model] is not None):
                diff = values[grpo_model] - values[orig_model]
                print(f"{diff:>+20.3f}", end='')
            print()
    
    # Summary statistics
    print("\n" + "="*100)
    print("SUMMARY STATISTICS")
    print("="*100)
    
    for condition_name, data_dict in [('With Answer', data_with), ('Without Answer', data_without)]:
        print(f"\n{condition_name}:")
        print("-" * 100)
        
        for source in sources:
            short_source = name_mapping.get(source, source[:20])
            values = [data['consistency_rate'] for data in data_dict[source].values()]
            
            if values:
                print(f"\n  {short_source}:")
                print(f"    Mean:   {np.mean(values):.3f}")
                print(f"    Median: {np.median(values):.3f}")
                print(f"    Std:    {np.std(values):.3f}")
                print(f"    Min:    {np.min(values):.3f}")
                print(f"    Max:    {np.max(values):.3f}")
    
    # Cross-source comparison
    if len(sources) == 2:
        print("\n" + "="*100)
        print("CROSS-SOURCE COMPARISON (Original vs GRPO)")
        print("="*100)
        
        short_a = name_mapping.get(sources[0], sources[0][:20])
        short_b = name_mapping.get(sources[1], sources[1][:20])
        cross_pair = f"{short_a} vs {short_b}"
        
        for condition_name, data_dict in [('With Answer', data_with), ('Without Answer', data_without)]:
            if cross_pair in data_dict[sources[0]]:
                consistency = data_dict[sources[0]][cross_pair]['consistency_rate']
                count = data_dict[sources[0]][cross_pair]['consistent_pairs']
                total = data_dict[sources[0]][cross_pair]['total_pairs']
                print(f"\n{condition_name}:")
                print(f"  {cross_pair}: {consistency:.3f} ({count}/{total})")
                print(f"  This measures how often the two source models give the same answer")
    
    # Effect analysis
    if grpo_model and orig_model:
        print(f"\n" + "="*100)
        print("EFFECT ANALYSIS")
        print("="*100)
        
        short_grpo = name_mapping.get(grpo_model, grpo_model[:20])
        short_orig = name_mapping.get(orig_model, orig_model[:20])
        
        print(f"\nEffect of Including Answer (with - without):")
        for source in sources:
            short_source = name_mapping.get(source, source[:20])
            common_pairs = set(data_with[source].keys()) & set(data_without[source].keys())
            
            if common_pairs:
                differences = [
                    data_with[source][pair]['consistency_rate'] -
                    data_without[source][pair]['consistency_rate']
                    for pair in common_pairs
                ]
                
                print(f"\n  {short_source}:")
                print(f"    Mean difference: {np.mean(differences):+.3f}")
                print(f"    Pairs improved:  {sum(1 for d in differences if d > 0)}/{len(differences)}")
                print(f"    Pairs worsened:  {sum(1 for d in differences if d < 0)}/{len(differences)}")
        
        print(f"\nEffect of GRPO ({short_grpo} - {short_orig}):")
        for condition_name, data_dict in [('With answer', data_with), ('Without answer', data_without)]:
            common_pairs = set(data_dict[grpo_model].keys()) & set(data_dict[orig_model].keys())
            
            if common_pairs:
                differences = [
                    data_dict[grpo_model][pair]['consistency_rate'] -
                    data_dict[orig_model][pair]['consistency_rate']
                    for pair in common_pairs
                ]
                
                print(f"\n  {condition_name}:")
                print(f"    Mean difference: {np.mean(differences):+.3f}")
                print(f"    Pairs improved:  {sum(1 for d in differences if d > 0)}/{len(differences)}")
                print(f"    Pairs worsened:  {sum(1 for d in differences if d < 0)}/{len(differences)}")


def main(folder_path, source_models=None, target_models=None):
    """Main function to run the consistency analysis."""
    print("Starting Comprehensive Llama Pairwise Consistency Analysis...")
    print(f"Analyzing folder: {folder_path}")
    print("-" * 100)
    
    result = load_data(folder_path, filter_source_models=source_models,
                      filter_target_models=target_models)
    
    if result is None:
        print("No valid JSONL files found.")
        return None
    
    data_by_source, model_names = result
    
    if not data_by_source['with'] and not data_by_source['without']:
        print("No valid data loaded.")
        return None
    
    total_files = len(data_by_source['with']) + len(data_by_source['without'])
    print(f"\nLoaded {total_files} model pair combinations "
          f"({len(data_by_source['with'])} with answer, {len(data_by_source['without'])} without answer)")
    print("-" * 100)
    
    # Calculate pairwise consistency
    print("\nCalculating pairwise consistency...")
    print("\nWith Answer:")
    pairwise_with = calculate_pairwise_consistency(data_by_source['with'])
    print("\nWithout Answer:")
    pairwise_without = calculate_pairwise_consistency(data_by_source['without'])
    
    if not pairwise_with and not pairwise_without:
        print("No pairwise consistency data found.")
        return None
    
    # Create visualizations
    pdf_pages = PdfPages('llama_comprehensive_analysis.pdf')
    print("\nGenerating visualizations...")
    plot_comprehensive_comparison(pairwise_with, pairwise_without,
                                 save_plots=True, pdf_pages=pdf_pages)
    pdf_pages.close()
    print(f"\n✓ Plots saved to: llama_comprehensive_analysis.pdf")
    
    # Print detailed comparison
    print_detailed_comparison(pairwise_with, pairwise_without)
    
    return pairwise_with, pairwise_without, model_names


if __name__ == "__main__":
    folder_path = "../outputs"
    
    results = main(folder_path, source_models=SOURCE_MODELS, target_models=TARGET_MODELS)
    
    if results:
        print("\n" + "="*100)
        print("Analysis completed successfully!")
        print("="*100)
    else:
        print("Analysis failed - check folder path and file formats.")